# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane 2: Refresh / Content Opportunity Scoring.

I'm picking this over the other three core lanes because the starter
playground already proves there's real signal here before I commit
7 weeks to it — the random forest beats the hand-written baseline
0.740 vs 0.240 on Precision@50 (verified below). I prefer it over
Lane 1 (Ranking Signal Analysis) because it ends in an actionable
ranked queue, not just an observational report, and over Lane 3/4
because content opportunity scoring maps directly onto a real
capacity-constrained decision: a small content team choosing what
to review this week out of thousands of pages.

In [8]:
lane = "Lane 2: Refresh / Content Opportunity Scoring"
print(f"Chosen lane: {lane}")


Chosen lane: Lane 2: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision improved: which pages a capacity-constrained content team
reviews first out of thousands of candidates.

Who acts: a content strategist, who pulls the top N pages from the
ranked queue into this week's refresh sprint instead of choosing by
gut feel or a naive "hasn't been touched in 6 months" rule.

Cost of a wrong recommendation:
- False positive (flagged, doesn't actually need review) -> wasted
  reviewer hours on a healthy page.
- False negative (needs review, not flagged) -> a real decline goes
  unnoticed and compounds silently.
Since reviewer time is the scarce resource, false negatives are
likely costlier, which argues for weighting recall within the top-K,
not just raw precision.

Scope limit: "the right page to fix" means the right page to review
first — not a guarantee that refreshing it causes recovery. Proving
causation would need an experiment this data alone can't give me.

In [9]:
decision = "which pages get reviewed first, given limited weekly capacity"
action = "strategist pulls top N pages into this week's refresh sprint"
cost_of_error = "false negative (missed decliner) > false positive (wasted review)"
print(decision, "\n", action, "\n", cost_of_error)

which pages get reviewed first, given limited weekly capacity 
 strategist pulls top N pages into this week's refresh sprint 
 false negative (missed decliner) > false positive (wasted review)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter dataset that make this lane worth
the next 7 weeks: the current label split, the pipeline's proven
lift over a hand rule, and the size of the population this lane's
reason codes actually target.

In [10]:
import subprocess, sys

# Regenerate outputs/model_results.json by running the reference pipeline
result = subprocess.run([sys.executable, "scripts/run_all.py"], capture_output=True, text=True)
print(result.stdout[-1500:])  # show the tail of the run so you can confirm it finished
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: current starter label distribution (the beginner proxy label)
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"Share labeled 'declining' (trend_direction == down): {declining_rate:.1%}")

# Number 2: verified pipeline results — evidence this lane is worth pursuing
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline Precision@50: {base:.3f}  |  Random forest Precision@50: {rf:.3f}")
print(f"Model beats baseline by {rf/base:.1f}x on this starter slice (30k rows, client-holdout)")

# Number 3: size of the population this lane's 'stale_visible_page' reason code targets
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"stale_visible_page candidates: {len(stale_visible)} of {len(df)} "
      f"({len(stale_visible)/len(df):.1%})")

ata/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /flyrank-ml-internship/outputs/refresh_queue.csv
Wrote model report: /flyrank-ml-internship/outputs/model_report.md
Wrote charts in: /flyrank-ml-internship/outputs/charts

▶ Step 5/5 — Report — a shareable PDF summary

STDERR: Traceback (most recent call last):
  File "/flyrank-ml-inte

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

I can say: this produces an observed, directional ranked queue of
review candidates, validated with client-holdout splitting, that
beats a transparent hand rule at Precision@50 on this starter slice.

I cannot say: that a flagged page is guaranteed to recover if
refreshed (needs a causal experiment), that this result holds at the
full ~79M-row warehouse scale (0.740 Precision@50 is from 30k rows
only), or that I've discovered a Google ranking factor.

I will not use FlyRank's own product decision flags (health_score,
priority_score, action_type) as model features or labels if I move
to the warehouse release — they aren't shipped in this data on
purpose, to avoid the circular-result trap. If I ever try to
reproduce one of those rules, I'll frame it explicitly as "can I
reproduce the existing rule?" rather than as discovery.

Every claim in the final write-up stays observed / measured /
directional / decision-support — never causal proof, never
"predicting Google."

In [11]:
unsafe_terms = ["url", "domain", "client_name", "query", "title"]
flagged_cols = [c for c in df.columns if any(term in c.lower() for term in unsafe_terms)]
print("Columns flagged for review:", flagged_cols if flagged_cols else "none — all columns are pseudonymized/observable")

Columns flagged for review: none — all columns are pseudonymized/observable


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.